In [0]:
import json
import os
import time
from datetime import datetime, timedelta, timezone

import requests

dbutils.widgets.text("api_key", "", "TMDB API key (optional; overrides the secret)")
dbutils.widgets.text(
    "raw_output_dir",
    "/Volumes/workspace/tmdb_lakehouse/raw/incremental",
    "Base folder for fetched batches",
)
dbutils.widgets.text("lookback_days", "7", "Change window in days (1-14)")
dbutils.widgets.text("max_movies", "300", "Maximum movies to fetch")
dbutils.widgets.text("batch_id", "", "Batch ID (blank = incremental-YYYYMMDD)")
dbutils.widgets.dropdown("run_silver", "false", ["true", "false"], "Run Silver after Bronze?")
dbutils.widgets.dropdown("run_gold", "true", ["true", "false"], "Run Gold after Silver?")

cfg = {
    name: dbutils.widgets.get(name).strip()
    for name in (
        "api_key", "raw_output_dir", "lookback_days",
        "max_movies", "batch_id", "run_silver", "run_gold",
    )
}

if not cfg["raw_output_dir"]:
    raise ValueError("raw_output_dir is required")

lookback_days = int(cfg["lookback_days"] or "7")
if not 1 <= lookback_days <= 14:
    raise ValueError("lookback_days must be between 1 and 14; TMDB rejects wider windows")

max_movies = int(cfg["max_movies"] or "300")
run_silver = cfg["run_silver"] == "true"
run_gold = cfg["run_gold"] == "true"

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.tmdb_lakehouse")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.tmdb_lakehouse.raw")


def resolve_api_key():
    if cfg["api_key"]:
        return cfg["api_key"]
    try:
        return dbutils.secrets.get(scope="tmdb-lakehouse-secrets", key="tmdb-api-key")
    except Exception:
        pass
    if os.environ.get("TMDB_API_KEY"):
        return os.environ["TMDB_API_KEY"]
    raise ValueError(
        "No TMDB API key. Store it as tmdb-lakehouse-secrets/tmdb-api-key, "
        "set the api_key widget, or export TMDB_API_KEY."
    )


api_key = resolve_api_key()
base_url = "https://api.themoviedb.org/3"
print("Configured:", {name: value for name, value in cfg.items() if name != "api_key"})

In [0]:
now = datetime.now(timezone.utc)
window_start = (now.date() - timedelta(days=lookback_days)).isoformat()
window_end = now.date().isoformat()
batch_id = cfg["batch_id"] or f"incremental-{now:%Y%m%d}"

changed_ids = []
page = 1
while True:
    response = requests.get(
        f"{base_url}/movie/changes",
        params={
            "api_key": api_key,
            "start_date": window_start,
            "end_date": window_end,
            "page": page,
        },
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()
    changed_ids.extend(item["id"] for item in payload.get("results", []))
    if page >= payload.get("total_pages", 1):
        break
    page += 1
    time.sleep(0.2)

# The changes feed can repeat an ID; keep the first occurrence and cap the sample.
changed_ids = list(dict.fromkeys(changed_ids))[:max_movies]
print(f"{len(changed_ids)} changed movie IDs between {window_start} and {window_end}")

movies, skipped = [], []
for position, movie_id in enumerate(changed_ids, 1):
    try:
        response = requests.get(
            f"{base_url}/movie/{movie_id}",
            params={"api_key": api_key, "append_to_response": "credits,keywords,reviews"},
            timeout=30,
        )
        response.raise_for_status()
        movies.append(response.json())
    except Exception as exc:
        skipped.append(movie_id)
        print("skip", movie_id, exc)
    time.sleep(0.05)
    if position % 50 == 0:
        print(f"{position}/{len(changed_ids)} fetched")

print(f"Fetched {len(movies)} movies; skipped {len(skipped)}")

In [0]:
if not movies:
    dbutils.notebook.exit(f"NO_CHANGES:{window_start}:{window_end}")

# One folder per batch keeps each Bronze payload reproducible for backfills.
batch_dir = f"{cfg['raw_output_dir'].rstrip('/')}/{batch_id}"
os.makedirs(batch_dir, exist_ok=True)
output_path = f"{batch_dir}/movies_incremental_load.json"

payload = {
    "load_type": "incremental",
    "window_start": window_start,
    "window_end": window_end,
    "record_count": len(movies),
    "records": movies,
}

with open(output_path, "w", encoding="utf-8") as handle:
    json.dump(payload, handle)

print(f"Wrote {len(movies)} records to {output_path}")

In [0]:
# Land the fetched snapshot in Bronze using the same widgets a manual run would use.
dbutils.notebook.run(
    "./phase2_raw_to_bronze",
    timeout_seconds=7200,
    arguments={
        "input_path": output_path,
        "load_type": "incremental",
        "batch_id": batch_id,
        "source_extracted_at": now.isoformat(),
        "window_start": window_start,
        "window_end": window_end,
    },
)
print("Bronze load finished for batch:", batch_id)

In [0]:
if not run_silver:
    print("Silver skipped. Run the Silver notebooks with batch_id =", batch_id)
    dbutils.notebook.exit(batch_id)

for notebook in (
    "./phase2_bronze_to_silver",
    "./phase2_silver_child_tables",
    "./phase2_silver_reviews",
):
    print("Running", notebook)
    dbutils.notebook.run(notebook, timeout_seconds=7200, arguments={"batch_id": batch_id})

if run_gold:
    print("Running ./phase3_gold_dimensional")
    dbutils.notebook.run(
        "./phase3_gold_dimensional",
        timeout_seconds=7200,
        arguments={"batch_id": batch_id},
    )
else:
    print("Gold skipped. Run phase3_gold_dimensional with batch_id =", batch_id)

print("Pipeline finished for batch:", batch_id)
dbutils.notebook.exit(batch_id)